# 🔁 再現性セットアップ（このノートの先頭で実行）

1. **① 依存ピン留め** を実行 → 「RESTART」を促されたら**ランタイム再起動**してから②へ
   （厳密再現が不要で現行 numpy で良ければ①はスキップ可）
2. **② ヘルパー定義** を実行 → `save_result(...)` が使えるようになる
3. ノート末尾で結果を保存（手転記の廃止・docs/71 §4）:
   ```python
   save_result("<このノート名>", metrics={...}, inputs=[csvパス...], seed=7,
               out_dir="/content/drive/MyDrive/forex_ml/results")
   ```


In [ ]:
# ① 依存ピン留め(再現性) — 実行後 RESTART を促されたら再起動してから先へ進む
# 厳密再現が不要なら、このセルはスキップして現行 numpy のまま回してもよい。
!pip install -q numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.2


In [ ]:
# ② 再現性ヘルパー: save_result を定義(metrics + 入力SHA-256 + 環境バージョンを JSON 保存)
import os, sys, json, hashlib, platform, datetime
try:
    _BASE = os.path.dirname(os.path.abspath(__file__))
except NameError:            # Colab/ノートでは __file__ が無い
    _BASE = os.getcwd()
RESULTS_DIR = os.path.join(_BASE, "results")
def _sha256(path, _b=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(_b), b""):
            h.update(c)
    return h.hexdigest()
def _env():
    out = {"python": sys.version.split()[0], "platform": platform.platform()}
    for m in ("numpy", "pandas", "matplotlib"):
        try: out[m] = getattr(__import__(m), "__version__", "?")
        except Exception: out[m] = None
    return out
def save_result(name, metrics, inputs=None, params=None, seed=None, out_dir=None):
    inputs = inputs or []
    d = out_dir or RESULTS_DIR
    rec = {"name": name,
           "saved_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
           "env": _env(), "seed": seed, "params": params,
           "inputs": [{"file": os.path.basename(p), "exists": os.path.exists(p),
                       "bytes": os.path.getsize(p) if os.path.exists(p) else None,
                       "sha256": _sha256(p) if os.path.exists(p) else None} for p in inputs],
           "metrics": metrics}
    os.makedirs(d, exist_ok=True)
    out = os.path.join(d, f"{name}.json")
    with open(out, "w") as f:
        json.dump(rec, f, ensure_ascii=False, indent=2, default=str)
    miss = [i["file"] for i in rec["inputs"] if not i["exists"]]
    print(f"[capture] {out}  inputs={len(inputs)}" + (f"  ★未検出={miss}" if miss else ""))
    return out
print("save_result 準備OK")


# edge7 — イントラデイ・セッション構造 探索（未踏フロンティア）

docs/27/28でdeferredだった「足内セッション構造」を事前登録N=108・Bonferroniで探索。
**結果: Bonferroni生存は USDCHF NYセッションSHORT 1件のみ・効果微小・2.76年＝LEAD未満**。
v8(USDCHF木曜)が10年で消えた前例と同型のため見送り。セッション単純ドリフトに新エッジ無し。

## 使い方
Colabなら下セルがrepoをcloneしYahoo H1(~2.76y)を取得して実行。10年H1(Dukascopy)があればそちらで再測を。
> ⚠ H1~2.76年=LEAD級。シミュレーションで将来保証なし。

In [ ]:
import os, sys, subprocess
REPO='chien-monitor'
if not os.path.exists('research/research_backtester.py'):
    if not os.path.exists(REPO):
        # 必要なら自分のフォークURLに変更。既にrepo内で実行するなら本ブロックは不要。
        try: subprocess.run(['git','clone','https://github.com/iq87jun-star/chien-monitor.git'],check=True)
        except Exception as e: print('clone skip:',e)
    if os.path.exists(REPO): os.chdir(REPO)
sys.path.insert(0,'research')
# データ(無ければYahooから取得: 日足10y + H1~2.76y)
if not os.path.exists('research/data/EURUSD_d.csv'):
    subprocess.run([sys.executable,'research/fetch_data.py'],check=False)
exec(open('research/edge7_intraday_sessions.py',encoding='utf-8').read())
